# Diagnóstico: por que o DBSCAN colapsa em 1 cluster + ruído?

Investigação da hipótese de que a alta dimensionalidade (PCA com muitos componentes) está achatando a distância euclidiana e fazendo o método do joelho escolher um `eps` grande demais. Nenhuma lógica de `utilitarios_agrupamentoManual.py` é alterada aqui -- só parâmetros já existentes (`USAR_PCA`, `NUMERO_MAXIMO_DE_COMPONENTES`, `K_VIZINHOS_EPS`) são variados, reaproveitando as mesmas funções do pipeline principal (`dbscan_regionaisManual.ipynb`). `estatisticas_da_curva` e `rodar_cenario_diagnostico` também vêm do módulo -- só reaproveitam funções já existentes do pipeline.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))


from utilitarios_agrupamentoManual import (
    BLOCOS_DE_TAXA, PASTA_TAXA,
    K_VIZINHOS_EPS, SEMENTE_ALEATORIA,
    LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA,
    montar_painel,
    descartar_colunas_sem_informacao_e_total, descartar_colunas_com_muitos_nulos, tratar_nulos_sem_imputar,
    padronizar_e_reduzir,
    estimar_eps_monko_kimura,
    estatisticas_da_curva, rodar_cenario_diagnostico,
    registrar_log,
)
from graficos_agrupamentoManual import grafico_k_distancias

## 0. Reconstrução da matriz de entrada (mesma etapa 1-2 do pipeline principal)

Reproduz `montar_painel` -> descarte de colunas -> `tratar_nulos_sem_imputar` de `dbscan_regionaisManual.ipynb` para chegar a `matriz`, e roda a etapa de eps padrão (`USAR_PCA=False`, sem redução) para obter `matriz_para_dbscan` e `resultado_eps` -- os mesmos usados como configuração "atual" nos cenários abaixo.

In [2]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)
matriz = painel
matriz = descartar_colunas_sem_informacao_e_total(matriz)
matriz = descartar_colunas_com_muitos_nulos(matriz, limiar_maximo=LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA)
matriz = tratar_nulos_sem_imputar(matriz)
registrar_log(f"Matriz final: {matriz.shape[0]} linha(s) x {matriz.shape[1]} atributo(s).")

_, matriz_para_dbscan, _, _ = padronizar_e_reduzir(matriz, usar_pca=False, semente=SEMENTE_ALEATORIA)
resultado_eps = estimar_eps_monko_kimura(matriz_para_dbscan, k=K_VIZINHOS_EPS, semente=SEMENTE_ALEATORIA)
figura_k_distancias = grafico_k_distancias(resultado_eps, k=K_VIZINHOS_EPS)

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.


[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).
[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Descartada(s) 13 coluna(s) com mais de 70% de nulos: ['casos-faixa_0-9', 'casos-escolaridade_nivel-superior-incompleto', 'mortalidade_0-9', 'mortalidade_10-19', 'mortalidade_20-29', 'mamografia_0-9', 'mamografia_10-19', 'mamografia_20-29', 'mamografia_30-39', 'mamografia_40-49', 'mamografia_50-59', 'mamografia_60-69', 'mamografia_70-ou-mais']
[LOG] Descartada(s) 945 de 1316 linha(s) (71.8%) por ter ao menos um valor ausente -- nenhum valor foi imputado.
[LOG] Matriz final: 371 linha

### 1. Curva bruta de k-distâncias na configuração atual (sem PCA)

Reaproveita `resultado_eps` calculado acima -- a mesma curva, agora com estatísticas descritivas para caracterizar a forma sem depender só da leitura visual do gráfico.

In [3]:
figura_k_distancias.show()

curva_atual = resultado_eps["curva"]
stats_atuais = estatisticas_da_curva(curva_atual)

print(f"Amplitude da curva: {curva_atual.min():.2f} a {curva_atual.max():.2f}")
print(f"Maior salto entre pontos consecutivos: {stats_atuais['salto_maximo']:.2f}, "
      f"em {stats_atuais['indice_maior_salto_pct']:.1%} da curva")
print(f"Razão salto-máximo / amplitude-total: {stats_atuais['razao_salto_amplitude']:.1%}")
print(f"Valor da curva nos primeiros 90% dos pontos: {curva_atual.min():.2f} a "
      f"{curva_atual[int(0.90*len(curva_atual))]:.2f}")
print(f"Valor da curva nos últimos 10% dos pontos: {curva_atual[int(0.90*len(curva_atual))]:.2f} a "
      f"{curva_atual.max():.2f}")

Amplitude da curva: 1.29 a 17.26
Maior salto entre pontos consecutivos: 4.67, em 98.7% da curva
Razão salto-máximo / amplitude-total: 29.2%
Valor da curva nos primeiros 90% dos pontos: 1.29 a 3.75
Valor da curva nos últimos 10% dos pontos: 3.75 a 17.26


### 2. Variação de NUMERO_MAXIMO_DE_COMPONENTES (10, 6, 4) e USAR_PCA=False

Mesma matriz de entrada (`matriz`) e mesmo `K_VIZINHOS_EPS` padrão -- só a redução de dimensionalidade muda. `USAR_PCA=False` roda o DBSCAN direto sobre os atributos padronizados, sem PCA nenhum.

In [4]:
linhas_diagnostico_pca = []

for n_componentes in (10, 6, 4):
    _, matriz_cenario, pca_cenario, variancia_cenario = padronizar_e_reduzir(
        matriz, numero_maximo_de_componentes=n_componentes, usar_pca=True, semente=SEMENTE_ALEATORIA,
    )
    linha = rodar_cenario_diagnostico(f"PCA={n_componentes}", matriz_cenario)
    linha["VARIANCIA_ACUMULADA"] = variancia_cenario[pca_cenario.n_components_ - 1]
    linhas_diagnostico_pca.append(linha)

linha_sem_pca = rodar_cenario_diagnostico("SEM PCA", matriz_para_dbscan)
linha_sem_pca["VARIANCIA_ACUMULADA"] = np.nan
linhas_diagnostico_pca.append(linha_sem_pca)

tabela_diagnostico_pca = pd.DataFrame(linhas_diagnostico_pca)
tabela_diagnostico_pca

[LOG] PCA aplicado: USAR_PCA=True (forçado; razão atributos/linhas = 0.059) -> 9 componente(s), 86.8% da variância acumulada.
[LOG] eps (Monko & Kimura, k=5, p=10, m=5, grau=3): 2.0643 no índice 283 de 371.


[LOG] MinPts escolhido: 4 (grade 3-36, passo 1).
[LOG] PCA aplicado: USAR_PCA=True (forçado; razão atributos/linhas = 0.059) -> 6 componente(s), 78.1% da variância acumulada.
[LOG] eps (Monko & Kimura, k=5, p=10, m=5, grau=3): 1.5191 no índice 287 de 371.


[LOG] MinPts escolhido: 4 (grade 3-24, passo 1).
[LOG] PCA aplicado: USAR_PCA=True (forçado; razão atributos/linhas = 0.059) -> 4 componente(s), 70.2% da variância acumulada.
[LOG] eps (Monko & Kimura, k=5, p=10, m=5, grau=3): 1.0068 no índice 287 de 371.


[LOG] MinPts escolhido: 5 (grade 3-16, passo 1).
[LOG] eps (Monko & Kimura, k=5, p=10, m=5, grau=3): 2.9709 no índice 280 de 371.


[LOG] MinPts escolhido: 3 (grade 3-88, passo 1).


,CENARIO,N_DIMENSOES,K_VIZINHOS_EPS,EPS,MIN_SAMPLES,N_CLUSTERS,N_RUIDO,SILHUETA_SEM_RUIDO,SILHUETA_COM_RUIDO,amplitude_curva,salto_maximo,razao_salto_amplitude,indice_maior_salto_pct,VARIANCIA_ACUMULADA
0,PCA=10,9,5,2.064274,4,2,67,0.219676,0.133708,16.261943,5.734661,0.352643,0.986523,0.867563
1,PCA=6,6,5,1.519057,4,2,59,0.145920,0.073267,16.393936,6.401895,0.390504,0.986523,0.781475
2,PCA=4,4,5,1.006841,5,2,73,0.323208,0.183724,16.423847,5.739047,0.349434,0.986523,0.701863
3,SEM PCA,22,5,2.970889,3,2,70,0.372746,0.261352,15.968496,4.668997,0.292388,0.986523,NaN


### 3. Variação de K_VIZINHOS_EPS (3, 5, 8), sem PCA

In [5]:
linhas_diagnostico_k = [
    rodar_cenario_diagnostico(f"k={k}", matriz_para_dbscan, k_eps=k)
    for k in (3, 5, 8)
]
tabela_diagnostico_k = pd.DataFrame(linhas_diagnostico_k)
tabela_diagnostico_k

[LOG] eps (Monko & Kimura, k=3, p=10, m=5, grau=3): 2.7898 no índice 274 de 371.


[LOG] ATENÇÃO: nenhum MinPts da grade produziu >= 2 agrupamentos válidos para a silhueta -- mantendo o menor MinPts testado.
[LOG] MinPts escolhido: 3 (grade 3-88, passo 1).
[LOG] eps (Monko & Kimura, k=5, p=10, m=5, grau=3): 2.9709 no índice 280 de 371.


[LOG] MinPts escolhido: 3 (grade 3-88, passo 1).
[LOG] eps (Monko & Kimura, k=8, p=10, m=5, grau=3): 3.1189 no índice 284 de 371.


[LOG] MinPts escolhido: 4 (grade 3-88, passo 1).


,CENARIO,N_DIMENSOES,K_VIZINHOS_EPS,EPS,MIN_SAMPLES,N_CLUSTERS,N_RUIDO,SILHUETA_SEM_RUIDO,SILHUETA_COM_RUIDO,amplitude_curva,salto_maximo,razao_salto_amplitude,indice_maior_salto_pct
0,k=3,22,3,2.789840,3,1,80,NaN,0.293580,10.937817,3.775764,0.345203,0.986523
1,k=5,22,5,2.970889,3,2,70,0.372746,0.261352,15.968496,4.668997,0.292388,0.986523
2,k=8,22,8,3.118919,4,2,49,0.352977,0.278959,19.469228,4.869633,0.250119,0.991914


### Tabela comparativa completa

`indice_maior_salto_pct` perto de 1.0 em todo cenário indica que o maior salto da curva está sempre nos últimos pontos (cauda de outliers), não no meio -- ausência de cotovelo nítido é, aqui, um resultado tão válido quanto um cotovelo nítido, e não deve ser forçado.

`SILHUETA_SEM_RUIDO` e `SILHUETA_COM_RUIDO` vêm de `resumir_agrupamento` -- a segunda versão trata o ruído como um cluster próprio e distorce a métrica, é reportada só para comparação (ver ressalva na Etapa 6 do pipeline principal).

In [6]:
tabela_diagnostico_completa = pd.concat([tabela_diagnostico_pca, tabela_diagnostico_k], ignore_index=True)
tabela_diagnostico_completa

,CENARIO,N_DIMENSOES,K_VIZINHOS_EPS,EPS,MIN_SAMPLES,N_CLUSTERS,N_RUIDO,SILHUETA_SEM_RUIDO,SILHUETA_COM_RUIDO,amplitude_curva,salto_maximo,razao_salto_amplitude,indice_maior_salto_pct,VARIANCIA_ACUMULADA
0,PCA=10,9,5,2.064274,4,2,67,0.219676,0.133708,16.261943,5.734661,0.352643,0.986523,0.867563
1,PCA=6,6,5,1.519057,4,2,59,0.145920,0.073267,16.393936,6.401895,0.390504,0.986523,0.781475
2,PCA=4,4,5,1.006841,5,2,73,0.323208,0.183724,16.423847,5.739047,0.349434,0.986523,0.701863
3,SEM PCA,22,5,2.970889,3,2,70,0.372746,0.261352,15.968496,4.668997,0.292388,0.986523,NaN
4,k=3,22,3,2.789840,3,1,80,NaN,0.293580,10.937817,3.775764,0.345203,0.986523,NaN
5,k=5,22,5,2.970889,3,2,70,0.372746,0.261352,15.968496,4.668997,0.292388,0.986523,NaN
6,k=8,22,8,3.118919,4,2,49,0.352977,0.278959,19.469228,4.869633,0.250119,0.991914,NaN
